#### Imports

In [8]:
import pandas as pd
import numpy as np
import time
import pyodbc
import pickle
import warnings
import os
from tensorflow.keras.models import load_model

warnings.filterwarnings('ignore')   
print("--- INITIALIZING FACTORY INFERENCE ENGINE ---")

--- INITIALIZING FACTORY INFERENCE ENGINE ---


#### Define the Target Serial

In [9]:
SELECTED_MACHINE = input("Enter the Machine Serial Number to analyze (e.g., 0521760): ").strip()
print(f"✅ Target Machine Set To: {SELECTED_MACHINE}")

✅ Target Machine Set To: 0521760


#### Azure SQL & Path Configuration

In [10]:
# 1. AZURE SQL CONFIGURATION
AZURE_SERVER = 'kreseakreiotprdsrv.database.windows.net'
AZURE_DATABASE = 'KRESEAKREIOTPRD'
AZURE_USERNAME = 'IOTAdmin'
AZURE_PASSWORD = 'oKuvodump5JNG7dM' 
AZURE_TABLE_NAME = 'MBP_ControllerData' 

# 2. PATHS TO TRAINED FILES
MODEL_PATH = r"C:\Users\DeelakaD\OneDrive - MAS Holdings (Pvt) Ltd\Documents\GitHub\Preventing-Mechanism\sewing_machine_predictive_lstm.h5"
SCALER_PATH = 'sewing_scaler.pkl'
ENCODER_PATH = 'sewing_encoder.pkl'

# 3. DATABASE DRIVER
driver = '{ODBC Driver 17 for SQL Server}'
conn_str = f'DRIVER={driver};SERVER={AZURE_SERVER};PORT=1433;DATABASE={AZURE_DATABASE};UID={AZURE_USERNAME};PWD={AZURE_PASSWORD}'

#### Load Baseline Model and Translators

In [11]:
try:
    model = load_model(MODEL_PATH)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    with open(ENCODER_PATH, 'rb') as f:
        encoder = pickle.load(f)
    print("✅ AI Model and Translators Loaded Successfully.")
except Exception as e:
    print(f"❌ ERROR LOADING FILES: {e}")

✅ AI Model and Translators Loaded Successfully.


#### Logic Engines

In [12]:
# Rule-Based Solution Engine
maintenance_solutions = {
    "Needle Breakage": "SOLUTION: Immediately replace the needle and check the needle guard alignment.",
    "Thread Jam": "SOLUTION: Clear the bobbin case, check thread tension, and clean the feed dogs.",
    "Blade Blunt": "SOLUTION: Schedule a replacement of the upper and lower cutting blades within 1 hour.",
    "High Foot Pressure": "SOLUTION: Reduce presser foot pressure dial by 2 turns.",
    "Healthy Baseline": "STATUS: Machine operating normally. No action required."
}

def parse_vibration_to_bands(vib_str):
    if pd.isna(vib_str): return {}
    parts = str(vib_str).split(',')
    res = {}
    try:
        for i in range(0, len(parts)-1, 2):
            f_start = int(parts[i])
            f_end = f_start + 10
            res[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
    except Exception:
        pass
    return res

# Define the exact 69 columns expected by the LSTM
electrical_cols = [ 
    'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
    'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax',
    'machinePowerMean', 'machinePowerMin', 'machinePowerMax'
]
vibration_bands = [f"{i}-{i+10}Hz" for i in range(10, 610, 10)]
ALL_69_FEATURES = electrical_cols + vibration_bands

#### The Live Monitoring Loop

In [ ]:
# To prevent processing of the same data, this will track the timestamp of the last processed record
last_processed_time = None

try:
    print(f"🔄 Connecting to Azure SQL Database: {AZURE_DATABASE}...")
    with pyodbc.connect(conn_str) as conn:
        print(f"✅ Monitoring Live Data Stream for Machine: {SELECTED_MACHINE}\n")
        
        # Query with Machine Filter
        query = f"""
            SELECT TOP 5 * FROM {AZURE_TABLE_NAME} 
            WHERE machineSerialNumber = '{SELECTED_MACHINE}' 
            ORDER BY dateTime DESC
        """
        
        while True:
            df_live = pd.read_sql(query, conn)
            
            if len(df_live) == 5:
                # Handle Time (UTC to SL Time)
                raw_db_time_utc = pd.to_datetime(df_live['dateTime'].iloc[0])
                sl_time = raw_db_time_utc + pd.Timedelta(hours=5, minutes=30)
                
                # Only process if data is fresh
                if sl_time != last_processed_time:
                    last_processed_time = sl_time
                    
                    # 1. Prepare Window (Reverse to Chronological)
                    df_live_sorted = df_live.iloc[::-1].reset_index(drop=True)
                    
                    # 2. Preprocess Features
                    vibs = pd.DataFrame(df_live_sorted['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
                    extras = df_live_sorted[electrical_cols]
                    df_window = pd.concat([extras, vibs], axis=1).fillna(0)
                    
                    # 3. Manual Feature Alignment (Fixes the StandardScaler error)
                    for col in ALL_69_FEATURES:
                        if col not in df_window.columns:
                            df_window[col] = 0
                    df_window = df_window[ALL_69_FEATURES] # Force correct order
                    
                    # 4. Scale and Reshape for LSTM
                    X_live_scaled = scaler.transform(df_window.values)
                    X_inference = np.array([X_live_scaled]) # Shape (1, 5, 69)
                    
                    # 5. AI Prediction
                    prediction_probs = model.predict(X_inference, verbose=0)
                    predicted_index = np.argmax(prediction_probs)
                    predicted_state = encoder.inverse_transform([predicted_index])[0]
                    
                    # 6. Actionable Output
                    print(f"--- [MACHINE: {SELECTED_MACHINE} | UPDATE: {sl_time.strftime('%H:%M:%S')}] ---")
                    print(f"⚠️ PREDICTED STATE: {predicted_state}")
                    
                    if predicted_state in maintenance_solutions:
                        print(f"🔧 {maintenance_solutions[predicted_state]}\n")
                    else:
                        print("🔧 STATUS: Checking for potential anomalies...\n")
            else:
                print(f"⏳ Waiting for data... Found {len(df_live)}/5 rows for {SELECTED_MACHINE}", end="\r")
            
            time.sleep(1)

except KeyboardInterrupt:
    print("\n🛑 Live monitoring stopped by user.")
except Exception as e:
    print(f"\n❌ CRITICAL ERROR: {e}")

🔄 Connecting to Azure SQL Database: KRESEAKREIOTPRD...
✅ Monitoring Live Data Stream for Machine: 0521760

--- [MACHINE: 0521760 | UPDATE: 16:19:53] ---
⚠️ PREDICTED STATE: Healthy Baseline
🔧 STATUS: Machine operating normally. No action required.

